# Practice: integrals, densities and minibatch noise
*AI from scratch, calculus 4* · lesson: https://ihor.sh/ai/c4-integrals.html

Areas by strips and by SciPy, probabilities as areas under the bell curve, Monte Carlo and its 1/√N error, and the
reason minibatch training works: a batch's gradient is a noisy guess at the full one, with noise shrinking like 1/√B.
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['f(x) gives all n heights at once; multiply by dx for the areas, then add them up with np.sum.', 'return np.sum(f(x)) * dx'], 2: ['Use integrate.quad(stats.norm.pdf, -1, 1)[0] to get the area between -1 and 1.', 'Use stats.norm.cdf(1) - stats.norm.cdf(-1) for the CDF formula.', 'Use 1 - stats.norm.cdf(2) to find the right-tail probability P(X > 2).'], 3: ['For each N in Ns, compute 200 sample estimates: np.mean(rng.standard_normal(N) ** 2).', 'Compute np.std of those 200 estimates for each N.', 'Store the list of standard deviations in `spread`.'], 4: ['Start with noise = [], then for each B append the mean of 300 calls of batch_distance(B).', 'A comprehension makes the 300: [batch_distance(B) for _ in range(300)]', 'noise = [], for B in Bs:, then indented: noise.append(np.mean([batch_distance(B) for _ in range(300)]))']}
_ANSWERS = {1: 'cmV0dXJuIG5wLnN1bShmKHgpKSAqIGR4', 2: 'd2l0aGluXzEgPSBpbnRlZ3JhdGUucXVhZChzdGF0cy5ub3JtLnBkZiwgLTEsIDEpWzBdCndpdGhpbl8xX2NkZiA9IHN0YXRzLm5vcm0uY2RmKDEpIC0gc3RhdHMubm9ybS5jZGYoLTEpCnBfdGFpbCA9IDEgLSBzdGF0cy5ub3JtLmNkZigyKQ==', 3: 'c3ByZWFkID0gW25wLnN0ZChbbnAubWVhbihybmcuc3RhbmRhcmRfbm9ybWFsKE4pICoqIDIpIGZvciBfIGluIHJhbmdlKDIwMCldKSBmb3IgTiBpbiBOc10=', 4: 'bm9pc2UgPSBbXQpmb3IgQiBpbiBCczoKICAgIG5vaXNlLmFwcGVuZChucC5tZWFuKFtiYXRjaF9kaXN0YW5jZShCKSBmb3IgXyBpbiByYW5nZSgzMDApXSkp'}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, stats

## 1. Area by strips
`midpoint(f, a, b, n)` cuts [a, b] into n strips of width `dx` = (b − a)/n; `x` holds the middle of every strip.
Finish it: each strip's area is f at its middle times `dx`, and the integral is the sum of the areas. Then compare
it with `integrate.quad` on $\int_0^2 e^{-x^2} dx$.

In [ ]:
def midpoint(f, a, b, n):
    """Numerical integration of f from a to b using the midpoint rule with n strips.
    
    Args:
        f: Callable function f(x) accepting float or array of floats.
        a: Float lower integration limit.
        b: Float upper integration limit.
        n: Integer number of subdivisions (strips).
    Returns:
        Float approximation of the definite integral.
    """
    dx = (b - a) / n
    x = a + (np.arange(n) + 0.5) * dx            # the middle of every strip, as one array
    # YOUR CODE HERE (stuck? hint(1), then answer(1))
    ...

g = lambda x: np.exp(-x ** 2)
exact = integrate.quad(g, 0, 2)[0]
print(exact, midpoint(g, 0, 2, 10), midpoint(g, 0, 2, 1000))

In [ ]:
check(1, lambda: abs(midpoint(g, 0, 2, 1000) - exact) < 1e-6, 
      "midpoint with 1000 strips should approximate the exact quad integral to within 1e-6.")
check(1, lambda: abs(midpoint(g, 0, 2, 10) - exact) / abs(midpoint(g, 0, 2, 100) - exact) > 50, 
      "Midpoint rule has O(h^2) error: 10x more strips should decrease error by roughly 100x (>50x).")

## 2. Probability is area
For the standard normal, set `within_1` = P(−1 < X < 1) by integrating its density `pdf` (`stats.norm.pdf`) with `quad`,
and `within_1_cdf` the same from the CDF `cdf`: `norm.cdf(1) − norm.cdf(−1)`. Then `p_tail` = P(X > 2).

In [ ]:
within_1 = within_1_cdf = p_tail = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print(within_1, within_1_cdf, p_tail)

In [ ]:
check(2, lambda: np.isclose(within_1, 0.6827, atol=1e-4) and np.isclose(within_1, within_1_cdf), 
      "About 68.27% of probability mass lies within 1 standard deviation for a standard normal.")
check(2, lambda: np.isclose(p_tail, 0.0228, atol=1e-4), 
      "P(X > 2) should be approximately 0.0228 (one-sided 2-sigma tail).")

## 3. Monte Carlo and its 1/√N error
Estimate $E[X^2] = 1$ for a standard normal by averaging $x^2$ over N random draws, 200 times for each N in `Ns`.
Set `spread`: the standard deviation of those 200 estimates for each N using `std` (`np.std`). On a log-log plot the slope should be −½.

In [ ]:
rng = np.random.default_rng(0)
Ns = [10, 100, 1000, 10_000]
spread = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
if spread is not None and len(spread):
    plt.loglog(Ns, spread, "o-", label="measured"); plt.loglog(Ns, np.sqrt(2) / np.sqrt(Ns), "--", label="√2/√N"); plt.legend(); plt.show()
print(spread)

In [ ]:
slope = lambda s: np.polyfit(np.log(Ns), np.log(s), 1)[0]
check(3, lambda: abs(slope(spread) + 0.5) < 0.1, 
      "Monte Carlo estimation error standard deviation falls at rate N^(-1/2) (slope on log-log plot ≈ -0.5).")

## 4. Minibatch gradients are noisy guesses
Linear regression on the diabetes data: the **full** gradient uses all 442 patients; a **minibatch** gradient uses
a random B of them. For each B in `Bs`, draw 300 minibatches and set `noise`: the average distance
$\lVert g_B - g_{\text{full}} \rVert$. It should fall roughly like $1/\sqrt{B}$.

In [ ]:
from sklearn.datasets import load_diabetes
d = load_diabetes()
X = np.column_stack([(d.data - d.data.mean(0)) / d.data.std(0), np.ones(len(d.data))]); y = d.target
w = np.zeros(X.shape[1])
grad = lambda Xb, yb: 2 / len(yb) * Xb.T @ (Xb @ w - yb)
g_full = grad(X, y)
Bs = [4, 16, 64, 256]

def batch_distance(B):
    "How far one random minibatch's gradient lands from the full gradient."
    i = rng.choice(len(y), B, replace=False)            # B different patients
    return np.linalg.norm(grad(X[i], y[i]) - g_full)

noise = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print(dict(zip(Bs, np.round(noise, 1))) if noise else noise)

In [ ]:
check(4, lambda: noise[0] / noise[2] > 3 and noise[1] / noise[3] > 3, 
      "Increasing batch size 16x should reduce gradient noise by ~4x (1/sqrt(B) scaling).")

## Next
Back to the path: https://ihor.sh/ai/ · next: probability 3, MLE and MAP.